# Study 892 — Corporate-Bond Ladder 🪜

**Does a held-to-maturity bond *ladder* beat a constant-maturity bond *fund*?**

The pitch you have heard: a bond **ladder** holds each rung to par and reinvests the cash
at the new yield, while a **fund** is "forced to sell falling bonds" and "locks in losses"
— *"so the ladder shines through a rate shock like 2022."* We race a duration-staggered
Treasury ladder (SHY/IEI/IEF/TLT) against the **AGG**/**BND** funds on total-return closes,
2007-06-30 → 2026-06-30 (229 months), excess of T-bill cash (BIL).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`4eeca3d56739`); the live cells run the fast synthetic control. All eight ETFs are
live survivors; the ~19-year joint window starts at BND/BIL inception.*


## 1. The idea — and the catch

The ladder story *sounds* airtight: hold each bond to maturity and you never realize a loss. But for a **default-free** bond that is an *accounting* story, not an *economic* one — the price that fell after rates rose **pulls back to par** by maturity, and that pull-to-par is the *exact reversal* of the mark-to-market loss the fund reported. Over a full horizon, two bond portfolios of the **same duration** earn the **same total return**. So the only real difference between a ladder and a fund is *how much duration* each carries.

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n_months': 229, 'fingerprint': '4eeca3d56739', 'ew_dur': 7.5, 'dm_dur': 6.0, 'agg_dur': 6.0, 'lad_ann': 3.02, 'lad_sharpe': 0.334, 'lad_ci': (-0.13, 0.81), 'lad_dd': -18.3, 'agg_ann': 3.07, 'agg_sharpe': 0.386, 'agg_ci': (-0.08, 0.9), 'agg_dd': -17.1, 'bnd_ann': 3.1, 'bnd_sharpe': 0.396, 'diff_ann': -0.01, 'diff_bps_mo': -0.11, 't_hac': -0.02, 't_1s': -0.02, 'diff_sharpe': -0.005, 'diff_sharpe_ci': (-0.48, 0.46), 'ew_ann': 2.92, 'ew_sharpe': 0.276, 'ew_dd': -23.2, 'era1': '2007-2015', 'era1_diff': 0.67, 'era1_t': 0.57, 'era2': '2016-2021', 'era2_diff': -0.6, 'era2_t': -0.87, 'era3': '2022-2026', 'era3_diff': -0.53, 'era3_t': -1.09, 'y2021_lad': -2.75, 'y2021_agg': -1.77, 'y2021_gap': -0.99, 'y2022_lad': -12.42, 'y2022_agg': -13.02, 'y2022_gap': 0.6, 'y2023_lad': 3.9, 'y2023_agg': 5.66, 'y2023_gap': -1.75, 'cost1': 0.9, 'net1': -0.02, 'tnet1': -0.04, 'cost2': 3.0, 'net2': -0.04, 'tnet2': -0.07, 'null_t_mean': 0.2, 'null_t_sd': 1.36, 'null_fire': 3, 'planted_recovered': 1.22, 'planted_t': 3.99}
print(f"duration-matched ladder ({R['dm_dur']}y) vs AGG ({R['agg_dur']}y):")
print(f"  ladder {R['lad_ann']:+.2f}%/yr, excess-Sharpe {R['lad_sharpe']:.3f}")
print(f"  AGG    {R['agg_ann']:+.2f}%/yr, excess-Sharpe {R['agg_sharpe']:.3f}")
print(f"  ladder - fund = {R['diff_ann']:+.2f}%/yr  (HAC t = {R['t_hac']:+.2f})")
print(f"  difference-Sharpe CI {R['diff_sharpe_ci']} straddles 0")

duration-matched ladder (6.0y) vs AGG (6.0y):
  ladder +3.02%/yr, excess-Sharpe 0.334
  AGG    +3.07%/yr, excess-Sharpe 0.386
  ladder - fund = -0.01%/yr  (HAC t = -0.02)
  difference-Sharpe CI (-0.48, 0.46) straddles 0


**A statistical dead heat.** Duration-matched, ladder minus fund is **-0.01%/yr** (HAC *t* = **-0.02**), and the difference-Sharpe confidence interval **(-0.48, 0.46)** sits right on top of zero. No held-to-maturity premium. If anything AGG edges the ladder on risk-adjusted terms, thanks to its credit/MBS diversification.

## 2. The ladder retail actually buys — it *loses*

An equal-weight SHY/IEI/IEF/TLT basket has a **7.5y** duration — 1.5y longer than AGG, because a quarter of it is 20-year TLT. That extra rate risk is not rewarded: excess-Sharpe **0.276** vs AGG's **0.386**, and a deeper **-23.2%** drawdown. Every apparent 'ladder edge' is a **duration bet in disguise**.

In [2]:
print(f"naive equal-weight ladder ({R['ew_dur']}y): {R['ew_ann']:+.2f}%/yr, "
      f"Sharpe {R['ew_sharpe']:.3f}, maxDD {R['ew_dd']:.1f}%")
print(f"AGG fund             ({R['agg_dur']}y): {R['agg_ann']:+.2f}%/yr, "
      f"Sharpe {R['agg_sharpe']:.3f}, maxDD {R['agg_dd']:.1f}%")

naive equal-weight ladder (7.5y): +2.92%/yr, Sharpe 0.276, maxDD -23.2%
AGG fund             (6.0y): +3.07%/yr, Sharpe 0.386, maxDD -17.1%


## 3. But didn't the ladder win 2022? For one year — then it gave it back

In calendar 2022 the pure-Treasury ladder *did* beat AGG by **+0.60 pp** (-12.42% vs -13.02%) — but not by holding to maturity: it dodged the **credit-spread widening** that hit AGG's corporate/MBS sleeve. And it **handed the whole thing back in 2023** (-1.75 pp) as spreads recovered. A one-year composition dodge, not a durable edge.

In [3]:
for yr,lad,agg,gap in [(2021,R['y2021_lad'],R['y2021_agg'],R['y2021_gap']),
                       (2022,R['y2022_lad'],R['y2022_agg'],R['y2022_gap']),
                       (2023,R['y2023_lad'],R['y2023_agg'],R['y2023_gap'])]:
    print(f"{yr}: ladder {lad:+.2f}%  AGG {agg:+.2f}%  gap {gap:+.2f} pp")

2021: ladder -2.75%  AGG -1.77%  gap -0.99 pp
2022: ladder -12.42%  AGG -13.02%  gap +0.60 pp
2023: ladder +3.90%  AGG +5.66%  gap -1.75 pp


## 4. A live synthetic control — the detector works, the market just says 'no'

We plant a ladder premium in a seeded toy world and check the detector recovers it (and stays silent on the null). This is the *machinery* proof — never market evidence. No network.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from bond_ladder import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(edge_annual=0.0, seed=892))
planted = st.synthetic_detect(data.synthetic_world(edge_annual=0.015, seed=892))
print('null world   : diff HAC t = %+.2f  (should be ~0)' % null['t_hac'])
print('planted +1.5%%: diff HAC t = %+.2f  (should light up)' % planted['t_hac'])

null world   : diff HAC t = -0.92  (should be ~0)
planted +1.5%: diff HAC t = +3.99  (should light up)


## The honest verdict

- **Signal — None.** Duration-matched, ladder minus fund is **-0.01%/yr** (HAC *t* = -0.02), CI on the difference-Sharpe (-0.48, 0.46) straddles zero, and the sign flips era to era (+0.67 / -0.60 / -0.53 %/yr). The naive ladder underperforms outright. HTM is an accounting illusion for default-free bonds.
- **Tradability — Mirage.** No gross edge exists, and the ETF ladder pays annual roll costs the one-ticker fund does not — net it is strictly behind. The ladder's real appeal is **behavioral** (no realized-loss statements, predictable cash flows), not a bankable risk-adjusted return.